# 06. Chronos-Bolt на национальных рядах СберИндекса (2018–2026)

На рядах МО у Chronos слишком короткая история (12–23 месяца). Здесь проверяем модель честно — на длинных
национальных рядах (с декабря 2018, около 94 месяцев) по пяти категориям трат.
Отсечки: каждый месяц с января 2023 по сентябрь 2025, прогноз на 12 месяцев вперёд.

**Запуск.** GPU не обязателен (рядов всего 5). Загрузите в файлы Colab
`consumer-spending_ru_1791179920805.parquet` и нажмите Run All. Результат — `chronos_national.parquet`.

In [ ]:
!pip -q install chronos-forecasting pyarrow

In [ ]:
import numpy as np
import pandas as pd
import torch
from chronos import BaseChronosPipeline

MODEL = "amazon/chronos-bolt-base"
CUTOFFS = pd.date_range("2023-01-01", "2025-09-01", freq="MS")
PRED_LEN, QUANTILES = 12, [0.1, 0.5, 0.9]
device = "cuda" if torch.cuda.is_available() else "cpu"

sp = pd.read_parquet("consumer-spending_ru_1791179920805.parquet")
sp["date"] = pd.to_datetime(sp.period)
y = sp.pivot_table(index="date", columns="type", values="value").sort_index()
print(y.shape, y.index.min().date(), y.index.max().date())

pipe = BaseChronosPipeline.from_pretrained(MODEL, device_map=device,
                                           torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32)

In [ ]:
rows = []
for T in CUTOFFS:
    hist = y[y.index <= T]
    ctx = [torch.tensor(hist[c].dropna().values, dtype=torch.float32) for c in y.columns]
    q, _ = pipe.predict_quantiles(ctx, prediction_length=PRED_LEN, quantile_levels=QUANTILES)
    q = q.float().cpu().numpy()
    ds = pd.date_range(T + pd.DateOffset(months=1), periods=PRED_LEN, freq="MS")
    for j, c in enumerate(y.columns):
        rows.append(pd.DataFrame({"series": c, "origin": T, "ds": ds,
                                  "q10": q[j, :, 0], "yhat": q[j, :, 1], "q90": q[j, :, 2]}))
fc = pd.concat(rows, ignore_index=True)
fc.to_parquet("chronos_national.parquet", index=False)
print(fc.shape, "→ скачайте chronos_national.parquet")